In [1]:
import QuantLib as ql

In [2]:
today = ql.Date(11, ql.December, 2012)
ql.Settings.instance().evaluationDate = today

<h4>First try</h4>

We start by instantiating helpers for all the rates used in the bootstrapping process, as reported in figure 25 of the paper.
The first three instruments are three 1-day deposit that give us discounting between today and the day after spot. They are modeled by three instances of the DepositRateHelper class with a tenor of 1 day and a number of fixing days going from 0 (for the deposit starting today) to 2 (for the deposit starting on the spot date).

In [3]:
helpers = [
    ql.DepositRateHelper(ql.QuoteHandle(ql.SimpleQuote(rate/100)),
                        ql.Period(1, ql.Days), fixingDays,
                        ql.TARGET(), ql.Following,
                        False, ql.Actual360())
    for rate, fixingDays in [(0.04, 0),(0.04, 1),(0.04, 2)] 
]

Then, we have a series of OIS (Overnight Index Swap) quotes for the first month. They are modeled by instances of the 
OISRateHelper class with varying tenors. They also require an instance of the Eonia class, which doesn’t need a forecast curve and can be shared between the helpers.

In [4]:
eonia = ql.Eonia()

In [7]:
helpers += [
    ql.OISRateHelper(2, ql.Period(*tenor),
                     ql.QuoteHandle(ql.SimpleQuote(rate/100)), eonia)
    for rate, tenor in [(0.070, (1,ql.Weeks)), (0.069, (2,ql.Weeks)),
                        (0.078, (3,ql.Weeks)), (0.074, (1,ql.Months))]

]